# English–Arabic Code-Switching Normalization

In [1]:
import pandas as pd
import re
from collections import Counter
import warnings

warnings.filterwarnings("ignore")

In [2]:
pd.set_option('display.max_colwidth', None)

test_df = pd.read_csv("test.csv")

df = pd.read_csv("train.csv")
df.head()

,Code Switched,Arabic
0,وقد عرض بارة أمام الجمعية العامة العناصر الرئيسية للمقاربة التي تتبعها في combating terrorism والنابعة من تقييم البلاد للـ threats و the challenges التي تطرحها هذه الآفة.,وقد عرض بارة أمام الجمعية العامة العناصر الرئيسية للمقاربة التي تتبعها في مكافحة الإرهاب والنابعة من تقييم البلاد للتهديدات و التحديات التي تطرحها هذه الآفة.
1,أظهرت the results بأن أكثر من ثلث الأزواج الذين تم إخضاعهم للفحص يعيشون حالة عداء فيما بينهم.,أظهرت النتائج بأن أكثر من ثلث الأزواج الذين تم إخضاعهم للفحص يعيشون حالة عداء فيما بينهم.
2,وثؤثث سهرة الاختتام الفنانة مناجاة يولشيفا من اوزباكستان التي اشتهرت بحسن ادائها للـ music الكلاسكية الاوزبكية ويرافقها في الـ singing شوكت مرزياف الذي discovered مواهبها الفنية.,وثؤثث سهرة الاختتام الفنانة مناجاة يولشيفا من اوزباكستان التي اشتهرت بحسن ادائها للموسيقى الكلاسكية الاوزبكية ويرافقها في الغناء شوكت مرزياف الذي اكتشف مواهبها الفنية.
3,وبحسب رأي the economic expert كميل الساري، فإن هذا هو أول لقاء في إطار the negotiations التي تقوم بها أي حكومة مع النقابات.,وبحسب رأي الخبير الاقتصادي كميل الساري، فإن هذا هو أول لقاء في إطار المفاوضات التي تقوم بها أي حكومة مع النقابات.
4,وأكد مكانة وسائل التجهيز فقال إنه لا يمكن لمعلم الـ music أن يعلم الـ pupil الموسيقى دون وجود بيانو مثلا، فهذه الوسائل لها دور اساسي.,وأكد مكانة وسائل التجهيز فقال إنه لا يمكن لمعلم الموسيقى أن يعلم التلميذ الموسيقى دون وجود بيانو مثلا، فهذه الوسائل لها دور اساسي.


In [3]:
print(f"Train len = {len(df)/1000}k")
print(f"Test len = {len(test_df)/1000}k")

Train len = 90.0k
Test len = 10.0k


## Text cleaning

In [4]:
df.isnull().sum()

Code Switched    0
Arabic           0
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
def check_df(df):
    for i in range(len(df)):
        for c in df.columns:
            text = df[c][i]

            # 1. Missing or empty sentences
            if pd.isna(text) or not str(text).strip():
                print(f"Row {i}, Column {c}: Missing/empty")
                continue

            # 2. Extra whitespace
            if re.search(r'\s{2,}', text) or text != text.strip():
                print(f"Row {i}, Column {c}: Extra whitespace")

            # 3. Consecutive duplicate words
            # Manually replaced \b with (?:^|\s) and (?=$|\s) because it counted diacritics as boundries
            duplicates = re.findall(r'(?:^|\s)(\w+)\s+\1(?=$|\s)', text, flags=re.IGNORECASE)
            if duplicates:
                print(f"Row {i}, Column {c}: Repeated words {duplicates}")

            # 4. Language validation
            if c == "Code Switched":
                if not re.search(r'[ء-ي]', text) or not re.search(r'[A-Za-z]', text):
                    print(f"Row {i}: Input missing Arabic or English")

            elif c == "Arabic":
                if not re.search(r'[ء-ي]', text) or re.search(r'[A-Za-z]', text):
                    print(f"Row {i}: Invalid Arabic target")

In [7]:
check_df(df)

In [8]:
check_df(test_df)

## Tokenization

### Word tokenizer

In [9]:
token = re.compile(
    r'[ء-يٱ][ء-يٱً-ٰٟـ]*'
    r'|[A-Za-z]+'
    r'|\d+(?:[.,/]\d+)*'
    r'|[^\w\s]',
    re.UNICODE
)

def tokenize(text):
    return token.findall(text)

In [10]:
df["input_tokens"] = df["Code Switched"].apply(tokenize)
df["target_tokens"] = df["Arabic"].apply(tokenize)

In [11]:
df["input_tokens"][0]

['وقد',
 'عرض',
 'بارة',
 'أمام',
 'الجمعية',
 'العامة',
 'العناصر',
 'الرئيسية',
 'للمقاربة',
 'التي',
 'تتبعها',
 'في',
 'combating',
 'terrorism',
 'والنابعة',
 'من',
 'تقييم',
 'البلاد',
 'للـ',
 'threats',
 'و',
 'the',
 'challenges',
 'التي',
 'تطرحها',
 'هذه',
 'الآفة',
 '.']

In [12]:
from tensorflow.keras.preprocessing.text import Tokenizer

input_tokenizer = Tokenizer(filters='', lower=False, oov_token='[oov]')
target_tokenizer = Tokenizer(filters='', lower=False, oov_token='[oov]')

input_tokenizer.fit_on_texts(df['input_tokens'])
target_tokenizer.fit_on_texts(df['target_tokens'])

print("Input vocabulary:", len(input_tokenizer.word_index) + 1)
print("Target vocabulary:", len(target_tokenizer.word_index) + 1)

Input vocabulary: 121076
Target vocabulary: 120499


### Subword tokenizer

In [13]:
import sentencepiece as spm

with open("tokenizer_training.txt", "w", encoding="utf-8") as f:
    for text in df["Code Switched"]:
        f.write(text + "\n")

    for text in df["Arabic"]:
        f.write(text + "\n")

spm.SentencePieceTrainer.train(
    input="tokenizer_training.txt",
    model_prefix="code_switch_bpe",
    model_type="bpe",
    vocab_size=8000,
    character_coverage=1.0,
    byte_fallback=True,
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
    normalization_rule_name="identity",
    remove_extra_whitespaces=False,
    add_dummy_prefix=False,
    shuffle_input_sentence=False,
    num_threads=1,
)

In [14]:
sp = spm.SentencePieceProcessor(model_file="code_switch_bpe.model")

In [15]:
print("Vocabulary size:", sp.vocab_size())

Vocabulary size: 8000


In [16]:
def test_bpe(df, name):
    sentences = df["Code Switched"].tolist() + df["Arabic"].tolist()

    mismatches = 0
    unknowns = 0

    for sentence in sentences:
        ids = sp.encode(sentence, out_type=int)

        if sp.decode(ids) != sentence:
            mismatches += 1

        unknowns += ids.count(sp.unk_id())

    print(f"{name} sentences tested: {len(sentences):,}")
    print(f"Reconstruction mismatches: {mismatches}")
    print(f"Unknown tokens: {unknowns}")
    print()

test_bpe(df, "Training")
test_bpe(test_df, "Testing")

Training sentences tested: 180,000
Reconstruction mismatches: 0
Unknown tokens: 0

Testing sentences tested: 20,000
Reconstruction mismatches: 0
Unknown tokens: 0



### Tokenization Choice

The original word-level tokenization produced vocabularies of 121,076 input tokens and 120,499 target tokens. Using SentencePiece BPE, we reduced this to a shared vocabulary of 8,000 tokens, approximately 6 times smaller.

BPE also reduces the risk of unknown words by splitting unfamiliar words into smaller subword units. Therefore, SentencePiece BPE is our final tokenization choice for the Seq2Seq model.